In [1]:
import sys
import importlib
import pandas as pd

if "../src" not in sys.path:
    sys.path.append("../src")

import preprocessing
import blocking

importlib.reload(preprocessing)
importlib.reload(blocking)

from preprocessing import add_normalized_columns
from blocking import add_blocking_keys, build_indexes, get_candidates

In [2]:
s1 = pd.read_csv(
    "../data/train/train_source1.tsv",
    sep="\t"
)

s2 = pd.read_csv(
    "../data/train/train_source2.tsv",
    sep="\t"
)

s3 = pd.read_csv(
    "../data/train/train_source3.tsv",
    sep="\t"
)

In [3]:
s1 = add_normalized_columns(s1)
s2 = add_normalized_columns(s2)
s3 = add_normalized_columns(s3)

In [4]:
s1 = add_blocking_keys(s1)
s2 = add_blocking_keys(s2)
s3 = add_blocking_keys(s3)

In [5]:
s2_indexes = build_indexes(s2)
s3_indexes = build_indexes(s3)


In [6]:
test_id = "S1-965667"

row = s1[s1["entity_id"] == test_id].iloc[0]

s2_candidates = get_candidates(row, s2_indexes)
s3_candidates = get_candidates(row, s3_indexes)

print("S1:", test_id)

print("\nS2 candidates:", len(s2_candidates))
print("S3 candidates:", len(s3_candidates))

print("\nS2:", list(s2_candidates)[:20])
print("\nS3:", list(s3_candidates)[:20])

S1: S1-965667

S2 candidates: 1745
S3 candidates: 1916

S2: ['S2-893506926', 'S2-76291616', 'S2-956496823', 'S2-893823254', 'S2-259001404', 'S2-627911089', 'S2-629134052', 'S2-41311108', 'S2-197956864', 'S2-404690741', 'S2-605238568', 'S2-125127027', 'S2-438694194', 'S2-768419960', 'S2-403355087', 'S2-858468054', 'S2-976117028', 'S2-861638650', 'S2-998379083', 'S2-144370856']

S3: ['S3-464336040', 'S3-395032825', 'S3-243056486', 'S3-861837507', 'S3-305491837', 'S3-656066439', 'S3-965753104', 'S3-602587847', 'S3-304847957', 'S3-475528786', 'S3-600991518', 'S3-81758946', 'S3-759847018', 'S3-452055184', 'S3-240318916', 'S3-769532659', 'S3-516865889', 'S3-542423744', 'S3-700639973', 'S3-30991138']


In [7]:
# Load ground truth
ground_truth = pd.read_csv(
    "../data/train/train_ground_truth.tsv",
    sep="\t"
)

# Get ground truth for our test entity
gt_row = ground_truth[
    ground_truth["source1_entity_id"] == test_id
].iloc[0]

# Convert comma-separated IDs into a set
matched_ids = str(gt_row["matched_entity_ids"])

if matched_ids == "nan" or matched_ids.strip() == "":
    gt_ids = set()
else:
    gt_ids = {
        x.strip()
        for x in matched_ids.split(",")
        if x.strip()
    }

# Combine S2 + S3 candidates
all_candidates = s2_candidates | s3_candidates

# Compare
found = gt_ids & all_candidates
missed = gt_ids - all_candidates

print("Ground truth matches:")
print(gt_ids)

print("\nTotal ground truth matches:", len(gt_ids))

print("\nFound in candidates:")
print(found)

print("\nNumber found:", len(found))

print("\nMissed:")
print(missed)

print("\nNumber missed:", len(missed))

print(
    "\nCandidate Recall:",
    f"{len(found) / len(gt_ids) * 100:.2f}%"
)

Ground truth matches:
{'S2-681193310', 'S3-11291185', 'S3-775321672', 'S2-743505751', 'S3-860443364'}

Total ground truth matches: 5

Found in candidates:
{'S2-681193310', 'S3-11291185', 'S3-775321672', 'S2-743505751', 'S3-860443364'}

Number found: 5

Missed:
set()

Number missed: 0

Candidate Recall: 100.00%


In [8]:
print("MATCHED CANDIDATES")
print("------------------")

for entity_id in sorted(found):
    if entity_id.startswith("S2-"):
        result = s2[s2["entity_id"] == entity_id]
    else:
        result = s3[s3["entity_id"] == entity_id]

    print("\n", result[
        [
            "entity_id",
            "business_name",
            "business_address",
            "country"
        ]
    ].to_string(index=False))

MATCHED CANDIDATES
------------------

    entity_id                business_name business_address country
S2-681193310 Maure Wilblims Colombier Inc              NaN      US

    entity_id            business_name business_address country
S2-743505751 Maure Williams Colombier              NaN      US

   entity_id              business_name                           business_address country
S3-11291185 maurewilliamscolombier.com Wayne Ave, Ticonderoga Townshiip, New York      US

    entity_id business_name                                 business_address country
S3-775321672       Dréxkor 85 Wanye Avenue, Ticonderoga Townshiip, New York      US

    entity_id             business_name business_address country
S3-860443364 Maure Williams Inc Center              NaN      US


In [9]:
sample_s1 = s1.sample(
    n=1000,
    random_state=42
)